# Lab 03: quantize a model

Lab: `corso/laboratori/lab-03-02-scala-quantizzazioni`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Quantize a model (lab 03-02)

The downloaded model is `Q8_0`. `--dry-run` estimates the result without writing; then we really quantize to `Q4_K_M`.

In [ ]:
%%bash
SRC=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/Qwen3.5-0.8B-Q8_0.gguf)
timeout 300 llama quantize --allow-requantize --dry-run $SRC Q4_K_M < /dev/null 2>&1 | tail -3

In [ ]:
%%bash
SRC=$(ls ~/.cache/huggingface/hub/models--ggml-org--Qwen3.5-0.8B-GGUF/snapshots/*/Qwen3.5-0.8B-Q8_0.gguf)
mkdir -p ~/llama-lab/quant
timeout 300 llama quantize --allow-requantize $SRC ~/llama-lab/quant/Qwen3.5-0.8B-Q4_K_M.gguf Q4_K_M $T < /dev/null 2>&1 | tail -4
ls -la $SRC ~/llama-lab/quant/

The quantized file is used like any other model.

In [ ]:
%%bash
timeout 300 llama cli -m ~/llama-lab/quant/Qwen3.5-0.8B-Q4_K_M.gguf -st -rea off --temp 0 -n 40 -t $T -p "Elenca tre città italiane sul mare." < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'